In [39]:
import pandas as pd

df = pd.read_csv("amazon_clean_READY.csv")

print(df.head())

   product_id                                       product_name  \
0  B07JW9H4J1  Wayona Nylon Braided USB to Lightning Fast Cha...   
1  B098NS6PVG  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...   
2  B096MSW6CT  Sounce Fast Phone Charging Cable & Data Sync U...   
3  B08HDJ86NZ  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...   
4  B08CF3B7N1  Portronics Konnect L 1.2M Fast Charging 3A 8 P...   

                                            category  discounted_price  \
0  Computers&Accessories|Accessories&Peripherals|...             399.0   
1  Computers&Accessories|Accessories&Peripherals|...             199.0   
2  Computers&Accessories|Accessories&Peripherals|...             199.0   
3  Computers&Accessories|Accessories&Peripherals|...             329.0   
4  Computers&Accessories|Accessories&Peripherals|...             154.0   

   actual_price  discount_percentage  rating  rating_count  \
0        1099.0                 64.0     4.2             0   
1         349.0       

In [40]:
# Clean price column
df["discounted_price"] = (
    df["discounted_price"]
    .astype(str)
    .str.replace(",", "")
    .str.replace("₹", "")
    .str.replace("Rs.", "")
    .str.strip()
)

df["discounted_price"] = pd.to_numeric(df["discounted_price"], errors="coerce")

# Clean rating
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Drop missing values
df = df.dropna(subset=["product_name", "category", "rating"])

In [41]:
print(type(df))

<class 'pandas.DataFrame'>


In [42]:
# Clean price column
df["discounted_price"] = (
    df["discounted_price"]
    .astype(str)
    .str.replace(",", "")
    .str.replace("₹", "")
    .str.replace("Rs.", "")
    .str.strip()
)

df["discounted_price"] = pd.to_numeric(df["discounted_price"], errors="coerce")

# Clean rating
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

# Drop missing values
df = df.dropna(subset=["product_name", "category", "rating"])

print("Cleaned:", df.shape)

Cleaned: (1337, 17)


In [43]:
df["features"] = df["category"] + " " + df["rating"].astype(str)

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["features"])

In [45]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [46]:
def get_recommendations(product_name, top_n=5):
    
    if product_name not in df["product_name"].values:
        return "Product not found"
    
    idx = df[df["product_name"] == product_name].index[0]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:top_n+1]
    
    product_indices = [i[0] for i in sim_scores]
    
    return df[["product_name", "discounted_price", "rating"]].iloc[product_indices]

In [47]:
df["product_name"].iloc[0]

'Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)'

In [48]:
print(get_recommendations("PASTE NAME HERE"))

Product not found


In [49]:
df["product_name"].head()

0    Wayona Nylon Braided USB to Lightning Fast Cha...
1    Ambrane Unbreakable 60W / 3A Fast Charging 1.5...
2    Sounce Fast Phone Charging Cable & Data Sync U...
3    boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...
4    Portronics Konnect L 1.2M Fast Charging 3A 8 P...
Name: product_name, dtype: str

In [50]:
"Wayona Nylon Braided USB to Lightning Fast Charging Cable"

'Wayona Nylon Braided USB to Lightning Fast Charging Cable'

In [51]:
print(get_recommendations("Wayona Nylon Braided USB to Lightning Fast Charging Cable"))

Product not found


In [52]:
product = df["product_name"].iloc[0]

print(get_recommendations(product))

                                        product_name  discounted_price  rating
1  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...             199.0     4.0
2  Sounce Fast Phone Charging Cable & Data Sync U...             199.0     3.9
3  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...             329.0     4.2
4  Portronics Konnect L 1.2M Fast Charging 3A 8 P...             154.0     4.2
5  pTron Solero TB301 3A Type-C Data and Fast Cha...             149.0     3.9


In [53]:
test_products = df["product_name"].sample(5).tolist()

for product in test_products:
    print("\n🔹 Recommendations for:", product)
    print(get_recommendations(product))


🔹 Recommendations for: MI REDMI 9i Sport (Carbon Black, 64 GB) (4 GB RAM)
                                          product_name  discounted_price  \
337  OnePlus Nord 2T 5G (Jade Fog, 8GB RAM, 128GB S...           28999.0   
338  OnePlus Nord 2T 5G (Gray Shadow, 8GB RAM, 128G...           28999.0   
339  Redmi A1 (Black, 2GB RAM, 32GB Storage) | Segm...            6499.0   
340  Redmi A1 (Light Green, 2GB RAM 32GB ROM) | Seg...            6499.0   
346  Samsung Galaxy M04 Dark Blue, 4GB RAM, 64GB St...            9499.0   

     rating  
337     4.3  
338     4.3  
339     4.0  
340     4.0  
346     4.2  

🔹 Recommendations for: MI 2-in-1 USB Type C Cable (Micro USB to Type C) 30cm for Smartphone, Headphone, Laptop (White)
                                        product_name  discounted_price  rating
1  Ambrane Unbreakable 60W / 3A Fast Charging 1.5...             199.0     4.0
2  Sounce Fast Phone Charging Cable & Data Sync U...             199.0     3.9
3  boAt Deuce USB 300 2 in 

In [54]:
test_products = df["product_name"].sample(10).tolist()

for product in test_products:
    print("\nRecommendations for:", product)
    print(get_recommendations(product))


Recommendations for: 7SEVEN® Compatible Lg Smart Tv Remote Suitable for Any LG LED OLED LCD UHD Plasma Android Television and AKB75095303 replacement of Original Lg Tv Remote Control
                                          product_name  discounted_price  \
55                           Tata Sky Universal Remote             230.0   
60   Airtel DigitalTV DTH Television, Setup Box Rem...             179.0   
79                                    Firestick Remote            1434.0   
96   LOHAYA Remote Compatible for Mi Smart LED TV 4...             249.0   
100  Dealfreez Case Compatible with Fire TV Stick 3...             349.0   

     rating  
55      3.7  
60      3.7  
79      4.0  
96      3.8  
100     4.3  

Recommendations for: Personal Size Blender, Portable Blender, Battery Powered USB Blender, with Four Blades, Mini Blender Travel Bottle for Juice, Shakes, and Smoothies (Pink)
                                           product_name  discounted_price  \
978    Orpat HHB-100E